# Football Managerial Decision Intelligence
### Interactive Player Performance, Starting XI Optimisation and Substitution Analysis Using Match Data
*Big Data Visualization — mini-project notebook*

| | |
|---|---|
| **Student name** | `[enter your name]` |
| **Registration number** | `[enter]` |
| **Course / section** | `[enter]` |
| **Faculty / guide** | `[enter]` |
| **Institution** | `[enter]` |
| **Project review (D4)** | 2-Nov-2026 · **Demo & viva (D2)** | 12-Nov-2026 |

**Research question.** *How can historical football match events, player performance indicators, lineup records and
match-state context be transformed into interactive visual analytics that support evidence-based starting-lineup and
substitution decisions?*

**Objective.** Build a reproducible pipeline — provenance → validation → features → analysis → decision support →
evaluation — over real open event data, and expose it through interactive Plotly visuals.

**Modules.** ① data collection & provenance · ② quality audit · ③ canonical tables · ④ feature engineering ·
⑤ exploratory analysis · ⑥ player intelligence · ⑦ team & tactical intelligence · ⑧ starting-XI decision support ·
⑨ substitution intelligence · ⑩ managerial decision patterns · ⑪ gated modelling · ⑫ findings, export, viva.

> **Scope and disclaimer.** This is football *performance-analysis decision support*. It is **not** a betting,
> fantasy or player-rating product. Every recommendation is a transparent heuristic computed only from information
> available before the decision, shown with its data coverage. Associations are never presented as causal effects,
> and the coaching staff — who hold information this dataset does not — make the final decision.

## 1 · Environment setup
Locates the project package (`src/football_analytics`), installs anything missing in Google Colab, fixes the
random seed, registers the plotting theme and enables widgets. In Colab, either set `PROJECT_REPO_URL` to your
GitHub copy of this repository, or upload `football_analytics_colab_bundle.zip` when prompted (see README).

In [1]:
import os, sys, subprocess, zipfile, importlib, platform, time, warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
HEADLESS = os.environ.get("FMA_HEADLESS") == "1"     # set by scripts/execute_notebook.py
PROJECT_REPO_URL = ""   # optional, e.g. "https://github.com/<you>/football-managerial-analytics.git"

def find_project_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path("/content/football-managerial-analytics"),
                  Path("/content/project"), Path("/content/drive/MyDrive/football-managerial-analytics")]
    candidates += [p.parent for p in Path("/content").glob("*/src/football_analytics")] if Path("/content").exists() else []
    for c in candidates:
        if (c / "src" / "football_analytics" / "__init__.py").exists():
            return c.resolve()
    return None

ROOT = find_project_root()
if ROOT is None and IN_COLAB:
    if PROJECT_REPO_URL:
        subprocess.run(["git", "clone", "-q", PROJECT_REPO_URL, "/content/football-managerial-analytics"], check=True)
    else:
        from google.colab import files
        print("Upload football_analytics_colab_bundle.zip (created by scripts/make_colab_bundle.py)")
        up = files.upload()
        zipfile.ZipFile(next(iter(up)), "r").extractall("/content/football-managerial-analytics")
    ROOT = find_project_root()
assert ROOT is not None, "Project root not found: run the notebook from the repository (see README)."
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

if IN_COLAB:  # Colab already ships pandas/numpy/plotly/sklearn/scipy; install only what is missing
    for mod, pkg in [("pyarrow", "pyarrow"), ("ipywidgets", "ipywidgets")]:
        if importlib.util.find_spec(mod) is None:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
    from google.colab import output
    output.enable_custom_widget_manager()

import numpy as np, pandas as pd
import plotly, plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
pio.renderers.default = "colab" if IN_COLAB else "notebook_connected+plotly_mimetype"

try:
    import ipywidgets as W
    WIDGETS_OK = True
except Exception:
    WIDGETS_OK = False

import football_analytics as fa
from football_analytics import (config as fcfg, ingestion, parsing, validation, features, player_analysis as pa,
                                tactical_analysis as ta, lineup_optimizer as lo, substitution_analysis as sa,
                                manager_analysis as ma, modeling as mdl, visualizations as viz, reporting,
                                data_sources, pipeline, advanced_visuals as av, dashboards as db)
SEED = 42
np.random.seed(SEED)
print(f"Project root : {ROOT}")
print(f"Runtime      : {'Google Colab' if IN_COLAB else 'local Jupyter'} | Python {platform.python_version()} | "
      f"pandas {pd.__version__} | plotly {plotly.__version__} | widgets {'on' if WIDGETS_OK else 'off'}")
print(f"Package      : football_analytics {fa.__version__}")

Project root : C:\Users\ASUS\Desktop\big data project
Runtime      : local Jupyter | Python 3.10.11 | pandas 2.1.4 | plotly 6.6.0 | widgets on
Package      : football_analytics 1.0.0


## 2 · Configuration
Every analytical choice lives here — nothing is hidden in later cells. Change a value and re-run from this cell.

* **Data mode** — `REAL_OPEN_DATA` (StatsBomb open data, default), `USER_DATA` (your own StatsBomb-format JSON in
  `data/external/user_statsbomb/`), or `DEMO_SYNTHETIC` (fictional league for offline testing; every chart then
  carries a red warning banner).
* **Competition / season** must be a pair that exists in the source catalogue shown in Section 3.
* `MATCH_LIMIT = None` analyses every published match; set e.g. `20` for a quick run (the truncation is reported).
* Thresholds, formation templates, weights, timing bands and substitution rules are editable dictionaries.

In [2]:
DATA_MODE      = fcfg.REAL_OPEN_DATA     # REAL_OPEN_DATA | USER_DATA | DEMO_SYNTHETIC
COMPETITION_ID = 9                       # 1. Bundesliga
SEASON_ID      = 281                     # 2023/2024
MATCH_LIMIT    = None                    # None = all published matches for the selection
FOCUS_TEAM     = "Bayer Leverkusen"      # team analysed in XI / substitution / manager sections
MIN_MINUTES    = 180                     # per-90 & percentile eligibility
OFFLINE        = False                   # True = use cached raw files only (no network)

if DATA_MODE == fcfg.DEMO_SYNTHETIC:     # the synthetic league has its own ids and no fixed focus team
    COMPETITION_ID, SEASON_ID, FOCUS_TEAM, MIN_MINUTES = 9999, 9999, None, 90

cfg = fcfg.Config(project_root=ROOT, data_mode=DATA_MODE, competition_id=COMPETITION_ID, season_id=SEASON_ID,
                  match_limit=MATCH_LIMIT, focus_team=FOCUS_TEAM, min_minutes=MIN_MINUTES,
                  min_group_size=3 if DATA_MODE == fcfg.DEMO_SYNTHETIC else 8)
BANNER = fcfg.SYNTHETIC_WARNING if cfg.is_synthetic else None

def status_badge():
    color = "#c8553d" if cfg.is_synthetic else "#2a9d8f"
    text = (f"DATA MODE: {cfg.data_mode} — {fcfg.SYNTHETIC_WARNING}" if cfg.is_synthetic else
            f"DATA MODE: {cfg.data_mode} — real StatsBomb open data (competition {cfg.competition_id}, "
            f"season {cfg.season_id})")
    display(HTML(f'<div style="background:{color};color:white;padding:8px 14px;border-radius:6px;'
                 f'font-weight:600;font-family:sans-serif">{text}</div>'))
status_badge()
display(pd.DataFrame({"setting": ["min_minutes", "min_group_size", "form_window", "reliability_full_minutes",
                                  "min_manager_matches", "impact_window_minutes"],
                      "value": [cfg.min_minutes, cfg.min_group_size, cfg.form_window, cfg.reliability_full_minutes,
                                cfg.min_manager_matches, cfg.impact_window_minutes]}))

,setting,value
0,min_minutes,180.0
1,min_group_size,8.0
2,form_window,5.0
3,reliability_full_minutes,900.0
4,min_manager_matches,10.0
5,impact_window_minutes,10.0


In [3]:
display(Markdown("**Starting-XI score weights** (components scaled 0-100, then × position compatibility)"))
display(pd.DataFrame([cfg.xi_score_weights]))
display(Markdown("**Substitution rules** — " + cfg.sub_rules["source"]))
display(pd.DataFrame([{k: v for k, v in cfg.sub_rules.items() if k != "source"}]))
display(Markdown("**Substitution timing bands** (clock minutes; stoppage time keeps counting, e.g. 45+2 = 47)"))
display(pd.DataFrame(cfg.sub_timing_bands))
display(Markdown("**Formation templates available:** " + ", ".join(cfg.formation_templates)))

**Starting-XI score weights** (components scaled 0-100, then × position compatibility)

,role_fit,recent_form,sample_reliability,workload_suitability,lineup_continuity
0,0.4,0.2,0.15,0.15,0.1


**Substitution rules** — User-configured default (IFAB Law 3 five-substitute option). Verify against the competition's regulations.

,max_substitutions,max_windows,extra_time_additional_substitutions,extra_time_additional_windows,interval_changes_use_window,window_grouping_seconds
0,5,3,1,1,False,90


**Substitution timing bands** (clock minutes; stoppage time keeps counting, e.g. 45+2 = 47)

,label,periods,min,max
0,0-30,[1],0,30
1,31-45+,[1],31,200
2,46-60,[2],0,60
3,61-70,[2],61,70
4,71-80,[2],71,80
5,81-90+,[2],81,200
6,Extra time,"[3, 4]",0,200


**Formation templates available:** 4-3-3, 4-2-3-1, 4-4-2, 3-5-2, 3-4-3, 3-4-2-1, 4-1-4-1

## 3 · Data collection
**Source:** [StatsBomb Open Data](https://github.com/statsbomb/open-data) (Hudl) — competitions, matches (with
managers, referee, stadium), lineups and event files, fetched as raw JSON over HTTPS. Files are cached unchanged in
`data/raw/`, every read is logged in a manifest (URL, UTC timestamp, bytes, SHA-256, status), and downloads use
timeouts, retries with back-off and graceful failure. Re-runs read from cache, so the notebook also works offline.

The catalogue below is what the source *actually* publishes; the competition/season in Section 2 must be one of
these rows.

In [4]:
if cfg.is_synthetic:
    ingestion.prepare_synthetic(cfg)
store = ingestion.DataStore(cfg, offline=OFFLINE)
catalogue = store.competitions()
print(f"Catalogue: {len(catalogue)} competition-seasons, {catalogue['competition_name'].nunique()} competitions")
sel = catalogue[(catalogue.competition_id == cfg.competition_id) & (catalogue.season_id == cfg.season_id)]
assert len(sel) == 1, "Selected competition/season not in catalogue — pick a row from the table below."
display(sel)
display(catalogue[["competition_id", "season_id", "competition_name", "season_name", "competition_gender",
                   "country_name", "match_available_360"]].head(80))

Catalogue: 80 competition-seasons, 24 competitions


,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_available,match_available_360
1,9,281,Germany,1. Bundesliga,male,False,False,2023/2024,2024-09-28T20:46:38.893391,2025-11-15T23:17:41.827093


,competition_id,season_id,competition_name,season_name,competition_gender,country_name,match_available_360
0,9,27,1. Bundesliga,2015/2016,male,Germany,None
1,9,281,1. Bundesliga,2023/2024,male,Germany,2025-11-15T23:17:41.827093
2,1267,107,African Cup of Nations,2023,male,Africa,2026-05-02T02:07:18.902396
3,16,276,Champions League,1970/1971,male,Europe,None
4,16,71,Champions League,1971/1972,male,Europe,None
...,...,...,...,...,...,...,...
75,35,75,UEFA Europa League,1988/1989,male,Europe,None
76,53,106,UEFA Women's Euro,2022,female,Europe,2026-05-05T03:05:32.480837
77,53,315,UEFA Women's Euro,2025,female,Europe,2026-04-27T22:03:28.087062
78,72,30,Women's World Cup,2019,female,International,None


In [5]:
if WIDGETS_OK and not HEADLESS:
    opts = [(f"{r.competition_name} — {r.season_name}", (r.competition_id, r.season_id)) for r in catalogue.itertuples()]
    pick = W.Dropdown(options=opts, value=(cfg.competition_id, cfg.season_id), description="Competition",
                      layout=W.Layout(width="520px"))
    out = W.Output()
    def _on(change):
        with out:
            out.clear_output()
            c, s = change["new"]
            print(f"Set COMPETITION_ID = {c}, SEASON_ID = {s} in Section 2 and re-run from there.")
    pick.observe(_on, names="value")
    display(pick, out)
else:
    print("Live competition picker available when running interactively (Colab/Jupyter).")

Live competition picker available when running interactively (Colab/Jupyter).


In [6]:
t0 = time.perf_counter()
matches_raw = store.matches_raw(cfg.competition_id, cfg.season_id)
selected = ingestion.select_matches(matches_raw, cfg.match_limit, cfg.focus_team)
print(f"Matches published for this competition-season: {len(matches_raw)}")
print(f"Matches selected for analysis: {len(selected)}"
      + (f" (focus team filter: {cfg.focus_team})" if cfg.focus_team else "")
      + (f" (limited to first {cfg.match_limit} chronologically)" if cfg.match_limit else " (no truncation)"))
files = store.match_files([m["match_id"] for m in selected], include_360=cfg.include_360)
manifest = store.manifest_frame()
print(f"Acquisition time: {time.perf_counter() - t0:.1f}s")
display(manifest.groupby(["resource", "status", "from_cache"]).agg(files=("relative_path", "size"),
                                                                   MB=("bytes", lambda b: round(b.sum() / 1e6, 2))))
display(manifest[["resource", "relative_path", "bytes", "sha256", "retrieved_utc", "status"]].head(6))

Matches published for this competition-season: 34
Matches selected for analysis: 34 (focus team filter: Bayer Leverkusen) (no truncation)


  fetched 10/34 matches


  fetched 20/34 matches


  fetched 30/34 matches


  fetched 34/34 matches
Acquisition time: 3.5s


,,,files,MB
resource,status,from_cache,,
competitions,ok,True,1,0.03
events,ok,True,34,119.22
lineups,ok,True,34,0.73
matches,ok,True,1,0.06


,resource,relative_path,bytes,sha256,retrieved_utc,status
0,competitions,competitions.json,34887,e6cd42f5d8956d6aa30fb917ce8d4c3b3df1879a93f02f...,2026-11-01T17:47:54+00:00,ok
1,matches,matches/9/281.json,63079,13dff90f126d9f73da410ae3292b2773744657d3ba968a...,2026-11-01T17:47:54+00:00,ok
2,events,events/3895052.json,3205861,ecc00b4e7ddd0154532329091d5d82fd7de4b9f6aa8da1...,2026-11-01T17:47:54+00:00,ok
3,lineups,lineups/3895052.json,22124,485e5e4f06136578a5dd18648397808110978cadb36a63...,2026-11-01T17:47:54+00:00,ok
4,events,events/3895060.json,3798711,1b2fcf16823fbf04f4ae40492d492304f8d54fdb991ceb...,2026-11-01T17:47:54+00:00,ok
5,lineups,lineups/3895060.json,24228,a94f051cea4356e4a09741cb737d5e6fd80b0b6611bdeb...,2026-11-01T17:47:54+00:00,ok


## 4 · Dataset inventory and collection evidence
Actual dimensions of what was collected and parsed (Dataset-collection rubric). Parsing turns nested JSON into
canonical tables; the figures below are computed, never typed in.

In [7]:
t0 = time.perf_counter()
bundle = pipeline.Bundle(config=cfg, store=store, catalogue=catalogue, matches_available=len(matches_raw))
pipeline.build_tables(bundle, selected, files)
T = bundle.tables
parse_seconds = time.perf_counter() - t0
scale = validation.dataset_scale(T, manifest, parse_seconds)
display(scale)
viz.coverage_cards(scale, banner=BANNER).show()

,measure,value
0,Competitions,1
1,Seasons,1
2,Matches listed by source (selected),34
3,Matches parsed,34
4,Events parsed,137765
5,Matchday-squad (lineup) rows,1353
6,Teams,18
7,Players (unique IDs in squads),460
8,Substitutions,301
9,Goals (periods 1-4),113


In [8]:
inv = ingestion.raw_inventory(cfg.raw_dir)
display(inv.groupby("folder").agg(files=("file", "size"), MB=("bytes", lambda b: round(b.sum() / 1e6, 2))))
m = T["matches"]
print(f"Date coverage: {m.match_date.min():%Y-%m-%d} → {m.match_date.max():%Y-%m-%d} "
      f"({m.match_date.dt.to_period('M').nunique()} calendar months)")
display(Markdown("**Source register and attribution**"))
display(data_sources.source_register()[["source", "role_in_project", "status"]])
display(Markdown(f"> {data_sources.STATSBOMB_ATTRIBUTION}" if not cfg.is_synthetic else f"> **{BANNER}**"))

,files,MB
folder,,
(root),1,0.03
events,34,119.22
lineups,34,0.73
matches,1,0.06


Date coverage: 2023-08-19 → 2024-05-18 (10 calendar months)


**Source register and attribution**

,source,role_in_project,status
0,StatsBomb Open Data (Hudl),"PRIMARY: competitions, matches (incl. managers...",used
1,API-Football v3,"OPTIONAL enrichment (coach, injuries, fixtures...",not used by default (no key supplied)
2,Metrica Sports sample data,OPTIONAL separate tracking demo only (anonymis...,not used
3,football-data.co.uk,OPTIONAL supplementary team-level results only,not used
4,User-supplied files,USER_DATA mode (StatsBomb-format JSON) and opt...,optional


> Data: StatsBomb Open Data (https://github.com/statsbomb/open-data), provided by StatsBomb / Hudl. Used for educational, non-commercial research.

**Big-data honesty.** The cards report the real scale: tens of MB of JSON and ~10⁵ events for one club-season —
a *scalable football-analytics pipeline / Big Data Visualization case study*, not "massive big data". Pandas is the
right tool at this volume (parsing + feature building take seconds). The same per-match design maps directly onto a
distributed engine: raw files are independent per `match_id`, so parsing is an embarrassingly parallel map
(e.g. Spark `binaryFiles` → `flatMap`), and every aggregation in Section 7 is a `groupBy` on `(match_id, team_id,
player_id)`. Spark is deliberately *not* added here because it would be decoration at this size — see the
processing times above and the scaling notes in the report.

## 5 · Data cleaning and quality audit
Domain rules are tested explicitly; nothing is dropped silently. Key parsing decisions (documented in
`parsing.py`): StatsBomb's lineup-file position spells are **not** reliably chronological, so on-pitch time is
rebuilt from the event stream (Starting XI → Substitution → sending-off), period-aware so first-half stoppage time
never overlaps the second half. Temporary "Player Off/On" treatment breaks are ignored.

In [9]:
bundle.quality, bundle.quality_log = validation.quality_checks(T)
display(bundle.quality)
n_flag = int((bundle.quality.status != "PASS").sum())
print(f"{len(bundle.quality) - n_flag}/{len(bundle.quality)} checks pass; {n_flag} flagged.")
display(Markdown("**Quality-action log** (retained / corrected / excluded / flagged records)"))
display(bundle.quality_log if len(bundle.quality_log) else pd.DataFrame({"action": ["no records excluded or corrected"]}))
display(Markdown("**Parser issue log**"))
display(T["parse_issues"] if len(T["parse_issues"]) else pd.DataFrame({"issue": ["none"]}))

,check,rule,violations,checked,status
0,Duplicate event IDs,event_id unique,0,137765,PASS
1,Duplicate squad rows,one row per player per match,0,1353,PASS
2,Event coordinates outside 120x80,"0<=x<=120, 0<=y<=80",0,136803,PASS
3,End coordinates outside pitch (+2 tolerance),end location on/near pitch,0,72499,PASS
4,"xG outside [0,1]",0<=statsbomb_xg<=1,0,916,PASS
5,Negative minutes played,minutes >= 0,0,1353,PASS
6,Matches with unordered event index,event index increasing,0,34,PASS
7,Clock steps backwards >5s within period,timestamps ordered (flag only),0,137765,PASS
8,Team-matches without exactly 11 starters,XI has 11 players,0,68,PASS
9,Team-matches without exactly one starting GK,one goalkeeper slot (event-position based),0,68,PASS


16/16 checks pass; 0 flagged.


**Quality-action log** (retained / corrected / excluded / flagged records)

,action
0,no records excluded or corrected


**Parser issue log**

,issue
0,none


In [10]:
display(validation.coverage_summary(T))
viz.missingness_heatmap(validation.event_type_coverage(T["events"]), banner=BANNER).show()

,coverage measure,value
0,Matches with lineup data (%),100.0
1,Squad rows with position (appeared players) (%),100.0
2,On-ball events with coordinates (%),100.0
3,Shots with xG (%),100.0
4,Passes with recipient (%),97.5
5,Matches with manager metadata (%),100.0
6,Matches with referee (%),100.0
7,Matches with stadium (%),100.0
8,Attendance / weather / injuries,not provided by source — not analysed


In [11]:
# Minutes reconstruction audit for one match: period bounds and every player's spell.
demo_mid = T["matches"]["match_id"].iloc[0]
display(T["period_bounds"][T["period_bounds"].match_id == demo_mid])
a = T["appearances"][T["appearances"].match_id == demo_mid]
display(a[["team_name", "player_name", "squad_status", "match_position", "minutes_played", "sub_on_minute",
           "sub_off_minute", "end_reason", "minutes_flag"]].sort_values(["team_name", "squad_status"]).round(1))

,match_id,period,start_clock,end_clock,duration,offset,bounds_source
0,3895052,1,0.0,2882.0,2882.0,0.0,half_start_end_events
1,3895052,2,2700.0,5749.0,3049.0,2882.0,half_start_end_events


,team_name,player_name,squad_status,match_position,minutes_played,sub_on_minute,sub_off_minute,end_reason,minutes_flag
1,Bayer Leverkusen,Granit Xhaka,starter,Left Defensive Midfield,98.8,NaN,NaN,final whistle,derived_from_events
2,Bayer Leverkusen,Jonathan Tah,starter,Center Back,98.8,NaN,NaN,final whistle,derived_from_events
4,Bayer Leverkusen,Lukáš Hrádecký,starter,Goalkeeper,98.8,NaN,NaN,final whistle,derived_from_events
5,Bayer Leverkusen,Jonas Hofmann,starter,Right Wing,91.3,NaN,88.0,substituted (Tactical),derived_from_events
7,Bayer Leverkusen,Álex Grimaldo,starter,Left Wing Back,98.8,NaN,NaN,final whistle,derived_from_events
8,Bayer Leverkusen,Odilon Kossonou,starter,Right Center Back,98.8,NaN,NaN,final whistle,derived_from_events
10,Bayer Leverkusen,Exequiel Palacios,starter,Right Defensive Midfield,98.8,NaN,NaN,final whistle,derived_from_events
11,Bayer Leverkusen,Edmond Fayçal Tapsoba,starter,Left Center Back,98.8,NaN,NaN,final whistle,derived_from_events
12,Bayer Leverkusen,Victor Boniface,starter,Center Forward,91.6,NaN,88.0,substituted (Tactical),derived_from_events
13,Bayer Leverkusen,Jeremie Frimpong,starter,Right Wing Back,80.4,NaN,77.0,substituted (Injury),derived_from_events


## 6 · Canonical tables
Source-native IDs are kept (never merged on names). Scores and match state are always computed from the team's
perspective; coordinates stay in one declared system (`statsbomb_120x80_attack_left_to_right`).

In [12]:
teams = pd.concat([T["matches"][["home_team_id", "home_team"]].set_axis(["team_id", "team_name"], axis=1),
                   T["matches"][["away_team_id", "away_team"]].set_axis(["team_id", "team_name"], axis=1)]
                  ).drop_duplicates().sort_values("team_name").reset_index(drop=True)
players = (T["appearances"].groupby(["player_id"]).agg(player_name=("player_name", "last"),
           team=("team_name", "last"), squads=("match_id", "nunique")).reset_index())
canon = {"competitions/seasons": catalogue[(catalogue.competition_id == cfg.competition_id)],
         "matches": T["matches"], "teams": teams, "players": players, "appearances (matchday squads)": T["appearances"],
         "events": T["events"], "substitutions": T["substitutions_raw"], "managers": T["managers"]}
display(pd.DataFrame([{"table": k, "rows": len(v), "columns": v.shape[1]} for k, v in canon.items()]))
display(T["matches"][["match_id", "match_date", "home_team", "home_score", "away_score", "away_team", "stadium",
                      "referee"]].head())
display(T["events"][["match_id", "period", "minute", "second", "elapsed_min", "type", "team_name", "player_name",
                     "x", "y", "end_x", "end_y", "outcome", "shot_xg"]].sample(6, random_state=SEED))
display(T["managers"].drop_duplicates(["team_name", "manager_name"]).head(8))

,table,rows,columns
0,competitions/seasons,2,10
1,matches,34,23
2,teams,18,2
3,players,460,4
4,appearances (matchday squads),1353,26
5,events,137765,54
6,substitutions,301,15
7,managers,68,8


,match_id,match_date,home_team,home_score,away_score,away_team,stadium,referee
0,3895052,2023-08-19,Bayer Leverkusen,3,2,RB Leipzig,BayArena,Felix Brych
1,3895060,2023-08-26,Borussia Mönchengladbach,0,3,Bayer Leverkusen,Stadion im BORUSSIA-PARK,Christian Dingert
2,3895067,2023-09-02,Bayer Leverkusen,5,1,Darmstadt 98,BayArena,Sven Jablonski
3,3895074,2023-09-15,Bayern Munich,2,2,Bayer Leverkusen,Allianz Arena,Daniel Schlager
4,3895086,2023-09-24,Bayer Leverkusen,4,1,FC Heidenheim,BayArena,Christian Dingert


,match_id,period,minute,second,elapsed_min,type,team_name,player_name,x,y,end_x,end_y,outcome,shot_xg
135887,3895348,2,45,35,48.583333,Carry,Bayer Leverkusen,Exequiel Alejandro Palacios,61.5,40.0,63.4,35.7,None,NaN
98389,3895258,2,79,34,83.433333,Ball Receipt*,FC Köln,Timo Hübers,44.4,75.8,NaN,NaN,None,NaN
80547,3895220,1,31,28,31.466667,Ball Receipt*,Bayer Leverkusen,Florian Wirtz,104.4,75.1,NaN,NaN,None,NaN
66238,3895194,1,2,3,2.050000,Ball Receipt*,Bayer Leverkusen,Granit Xhaka,42.4,59.8,NaN,NaN,None,NaN
101713,3895266,2,56,5,57.250000,Foul Won,Wolfsburg,Maxence Lacroix,6.3,65.9,NaN,NaN,None,NaN
63799,3895182,1,45,54,45.900000,Duel,Bochum,Erhan Mašović,5.3,37.0,NaN,NaN,None,NaN


,match_id,team_id,team_name,manager_id,manager_name,manager_full_name,manager_country,manager_source
0,3895052,904,Bayer Leverkusen,1000310,Xabi Alonso,Xabier Alonso Olano,Spain,StatsBomb matches.json
1,3895052,182,RB Leipzig,496,Marco Rose,Marco Rose,Germany,StatsBomb matches.json
2,3895060,185,Borussia Mönchengladbach,732,Gerardo Seoane,Gerardo Seoane Castro,Switzerland,StatsBomb matches.json
5,3895067,872,Darmstadt 98,100,Torsten Lieberknecht,Torsten Lieberknecht,Germany,StatsBomb matches.json
6,3895074,169,Bayern Munich,279,Thomas Tuchel,Thomas Tuchel,Germany,StatsBomb matches.json
9,3895086,201,FC Heidenheim,108,Frank Schmidt,Frank Schmidt,Germany,StatsBomb matches.json
10,3895095,177,FSV Mainz 05,4975,Bo Svensson,Bo Svensson,Denmark,StatsBomb matches.json
13,3895107,186,FC Köln,104,Steffen Baumgart,Steffen Baumgart,Germany,StatsBomb matches.json


## 7 · Feature engineering
Every metric has a definition, numerator/denominator, unit and source field (table below, exported to the data
dictionary). Rules applied:
* **Per-90** = count ÷ minutes × 90, only where minutes > 0 (no division by zero). Minutes are *actual playing time
  including stoppage*, so per-90 values are slightly lower than "nominal 90" conventions — consistently.
* **Role normalisation** — percentiles are computed within the player's role (if ≥ `min_group_size` eligible
  players) or role family otherwise, against the pool of players above `MIN_MINUTES`.
* **Project-defined approximations** (progressive pass/carry, possession proxy, field tilt, PPDA approximation) are
  labelled as such. xT, packing and tracking-based running metrics are **not** computed — the data cannot support them.
* **Workload proxies** (minutes in previous 7/14/28 days, rest days) count only matches in the dataset — they are not
  medical fatigue measures.

In [13]:
pipeline.build_features(bundle)
B = bundle
FOCUS = cfg.focus_team or B.team_stats["team_name"].value_counts().index[0]
print(f"Feature tables built in {B.timings['features_s']:.1f}s | player-match rows {len(B.pms)} | "
      f"player-season rows {len(B.season)} | team-match rows {len(B.team_stats)} | focus team: {FOCUS}")
display(features.metric_dictionary().head(45))

Feature tables built in 2.9s | player-match rows 1049 | player-season rows 377 | team-match rows 68 | focus team: Bayer Leverkusen


,metric,definition,numerator,denominator,unit,source_fields
0,minutes_played,Actual playing time on pitch incl. stoppage (p...,sum of spell lengths,-,minutes,"events: Starting XI, Substitution, Foul Commit..."
1,goals,Shots with outcome Goal in periods 1-4 (shoot-...,count,-,goals,shot.outcome
2,npxg,Sum of StatsBomb xG on non-penalty shots,sum xG,-,xG,"shot.statsbomb_xg, shot.type"
3,xg,Sum of StatsBomb xG on all shots (periods 1-4),sum xG,-,xG,shot.statsbomb_xg
4,xa,xG of shots assisted by the player's passes (p...,sum xG,-,xG,"pass.assisted_shot_id, shot.statsbomb_xg"
5,shots,Shot events (periods 1-4),count,-,shots,type=Shot
6,shots_on_target,Shots with outcome Goal / Saved / Saved To Post,count,-,shots,shot.outcome
7,assists,Passes flagged goal_assist by StatsBomb,count,-,assists,pass.goal_assist
8,key_passes,Passes flagged shot_assist or goal_assist,count,-,passes,"pass.shot_assist, pass.goal_assist"
9,passes,Pass attempts,count,-,passes,type=Pass


In [14]:
cols = ["player_name", "team_name", "primary_position", "primary_role", "appearances", "starts", "minutes_played",
        "goals", "xg", "xg_p90", "xa_p90", "key_passes_p90", "progressive_passes_p90", "pressures_p90",
        "pass_completion_pct", "benchmark_group", "benchmark_n", "coverage_label"]
display(B.season_pct[B.season_pct.team_name == FOCUS][cols].round(2).head(20))
display(Markdown("**Minimum-minutes sensitivity** — how the xG/90 leaderboard changes with the threshold"))
display(pa.threshold_sensitivity(B.season, "xg_p90"))
viz.threshold_sensitivity_chart(B.season, banner=BANNER).show()

,player_name,team_name,primary_position,primary_role,appearances,starts,minutes_played,goals,xg,xg_p90,xa_p90,key_passes_p90,progressive_passes_p90,pressures_p90,pass_completion_pct,benchmark_group,benchmark_n,coverage_label
0,Lukáš Hrádecký,Bayer Leverkusen,Goalkeeper,GK,33,33,3214.60,0.0,0.00,0.00,0.00,0.00,0.00,0.03,80.16,GK,14.0,high data coverage
1,Granit Xhaka,Bayer Leverkusen,Right Defensive Midfield,DM,33,32,3034.77,3.0,1.57,0.05,0.08,1.22,4.89,9.16,92.30,DM,14.0,high data coverage
2,Álex Grimaldo,Bayer Leverkusen,Left Wing Back,WB,33,31,2994.68,10.0,5.21,0.16,0.27,2.28,4.27,9.65,82.58,Defender,38.0,high data coverage
3,Jonathan Tah,Bayer Leverkusen,Center Back,CB,31,30,2848.12,4.0,1.81,0.06,0.01,0.16,2.02,6.83,94.92,CB,22.0,high data coverage
4,Florian Wirtz,Bayer Leverkusen,Left Attacking Midfield,AM,32,26,2528.87,11.0,8.47,0.30,0.28,2.53,7.15,23.17,83.81,Midfielder,22.0,high data coverage
5,Jeremie Frimpong,Bayer Leverkusen,Right Wing Back,WB,31,27,2378.47,9.0,8.72,0.33,0.17,1.63,2.76,13.13,79.09,Defender,38.0,high data coverage
6,Jonas Hofmann,Bayer Leverkusen,Right Attacking Midfield,AM,32,26,2334.68,4.0,5.71,0.22,0.30,2.93,3.39,20.97,81.09,Midfielder,22.0,high data coverage
7,Edmond Fayçal Tapsoba,Bayer Leverkusen,Left Center Back,CB,28,23,2262.52,0.0,1.01,0.04,0.03,0.28,4.06,8.00,91.62,CB,22.0,high data coverage
8,Odilon Kossonou,Bayer Leverkusen,Right Center Back,CB,22,21,1962.23,1.0,0.39,0.02,0.04,0.50,2.94,8.94,90.58,CB,22.0,high data coverage
9,Exequiel Palacios,Bayer Leverkusen,Left Defensive Midfield,DM,24,21,1949.78,4.0,2.69,0.12,0.15,1.25,4.62,15.79,92.33,DM,14.0,high data coverage


**Minimum-minutes sensitivity** — how the xG/90 leaderboard changes with the threshold

,min_minutes,eligible_players,top_5
0,90,224,"Victor Boniface (0.87), Serhou Guirassy (0.85)..."
1,180,84,"Victor Boniface (0.87), Serhou Guirassy (0.85)..."
2,270,19,"Victor Boniface (0.87), Adam Hložek (0.50), Pa..."
3,450,18,"Victor Boniface (0.87), Adam Hložek (0.50), Pa..."
4,900,17,"Victor Boniface (0.87), Patrik Schick (0.43), ..."


In [15]:
display(Markdown("**Workload proxies (pre-match, dataset matches only)** — focus team, latest match"))
last_mid = B.team_stats[B.team_stats.team_name == FOCUS].sort_values("match_date")["match_id"].iloc[-1]
w = B.workload[B.workload.match_id == last_mid].merge(
    T["appearances"][["match_id", "player_id", "player_name", "squad_status"]], on=["match_id", "player_id"])
display(w[["player_name", "squad_status", "minutes_prev_7d", "minutes_prev_14d", "minutes_prev_28d", "rest_days",
           "started_prev_match", "consecutive_starts_before", "start_share_before"]].round(2)
        .sort_values("start_share_before", ascending=False).head(12))
display(Markdown("**Team style indicators** (averages per analysed match)"))
display(ta.team_style_summary(B.team_stats).head(10))

**Workload proxies (pre-match, dataset matches only)** — focus team, latest match

,player_name,squad_status,minutes_prev_7d,minutes_prev_14d,minutes_prev_28d,rest_days,started_prev_match,consecutive_starts_before,start_share_before
2,Felix Uduokhai,starter,0.00,0.00,0.00,126.0,True,1,1.00
3,Jeffrey Gouweleeuw,starter,0.00,0.00,0.00,126.0,True,1,1.00
4,Niklas Dorsch,starter,0.00,0.00,0.00,126.0,True,1,1.00
5,Phillip Tietz,starter,0.00,0.00,0.00,126.0,True,1,1.00
1,Ermedin Demirović,starter,0.00,0.00,0.00,126.0,True,1,1.00
11,Iago,starter,0.00,0.00,0.00,126.0,True,1,1.00
19,Granit Xhaka,used_substitute,80.92,153.30,255.32,6.0,True,2,0.97
22,Lukáš Hrádecký,starter,96.27,192.27,394.05,6.0,True,14,0.97
25,Álex Grimaldo,starter,14.97,14.97,207.17,6.0,False,0,0.91
21,Jonathan Tah,starter,96.27,119.88,268.93,6.0,True,1,0.88


**Team style indicators** (averages per analysed match)

,team_name,matches,goals_for,goals_against,xg,xg_against,shots,shots_against,possession_proxy_pct,field_tilt_pct,pass_completion_pct,progressive_passes,pressures,ppda,set_piece_shots,points
0,Bayer Leverkusen,34,2.62,0.71,2.14,0.82,18.32,8.62,61.70,67.61,87.64,37.85,140.76,16.95,9.59,2.65
1,Augsburg,2,0.50,1.50,0.55,1.87,7.00,22.50,29.92,20.83,78.81,19.00,214.50,20.85,2.50,0.00
2,Werder Bremen,2,0.00,4.00,0.43,2.05,4.50,17.00,42.74,36.26,82.75,18.50,211.00,22.99,2.50,0.00
3,VfB Stuttgart,2,1.50,1.50,2.10,1.59,16.50,16.50,51.31,43.25,84.90,19.00,186.50,12.35,7.50,1.00
4,Union Berlin,2,0.00,2.50,0.29,2.22,5.50,19.00,33.21,29.13,77.20,16.00,200.50,39.03,3.50,0.00
5,RB Leipzig,2,2.00,3.00,1.40,1.70,12.00,14.50,47.88,40.04,82.89,22.00,169.50,13.61,4.50,0.00
6,Hoffenheim,2,1.50,2.50,1.07,2.02,9.00,22.00,39.92,28.93,79.87,19.50,178.00,25.06,3.00,0.00
7,Freiburg,2,1.50,2.50,0.85,2.01,10.50,18.50,31.62,28.55,76.51,16.00,132.00,20.84,8.00,0.00
8,FSV Mainz 05,2,0.50,2.50,0.79,0.95,11.50,11.50,39.42,40.91,77.96,18.00,169.00,16.63,8.00,0.00
9,FC Köln,2,0.00,2.50,0.94,2.12,6.50,20.00,29.36,16.87,77.29,10.50,202.00,36.50,3.00,0.00


## 8 · Exploratory data analysis
Opens with the **season dashboard** (KPI cards, points race against shot-based expected points, match-by-match xG),
then distributions (what is typical, what is extreme), position-group differences (RQ1), relationships (correlation
is association only), team trends, match dominance and a single-match dashboard.

*Expected points (xPts)* simulate each match 5,000 times, treating every shot as a Bernoulli trial with its
StatsBomb xG — a summary of chance quality, not of what "should" have happened (shots are not independent in reality).

In [16]:
xpts = ta.expected_points(B.flags, B.team_stats, n_sims=5000, seed=SEED)
season_fig = db.season_dashboard(B.team_stats, xpts, FOCUS, banner=BANNER)
season_fig.show()
fx = xpts[xpts.team_name == FOCUS]
display(Markdown(f"**Observation (computed):** {FOCUS} collected **{fx.points.sum():.0f} points** against "
                 f"**{fx.xpts.sum():.1f} expected points** from shot quality "
                 f"({fx.points.sum() - fx.xpts.sum():+.1f}); the gap can reflect finishing, goalkeeping, game-state "
                 "management and chance — it is not a measure of luck alone."))

**Observation (computed):** Bayer Leverkusen collected **90 points** against **74.8 expected points** from shot quality (+15.2); the gap can reflect finishing, goalkeeping, game-state management and chance — it is not a measure of luck alone.

In [17]:
eda_metrics = ["xg_p90", "key_passes_p90", "progressive_passes_p90", "progressive_carries_p90", "pressures_p90",
               "interceptions_p90", "tackles_won_p90", "pass_completion_pct", "box_touches_p90", "aerials_won_p90"]
elig = B.season_pct[B.season_pct.eligible]
viz.feature_distributions(elig, eda_metrics, title=f"Feature distributions (eligible players, n={len(elig)})",
                          banner=BANNER).show()
viz.role_distribution(B.season_pct, eda_metrics, cfg.min_minutes, banner=BANNER).show()

In [18]:
rd = elig.groupby("role_family")[["xg_p90", "progressive_passes_p90", "interceptions_p90", "pass_completion_pct",
                                   "pressures_p90"]].median().round(2)
rd.insert(0, "players", elig.groupby("role_family").size())
display(Markdown("**Observation (computed): median per-90 profile by role family**"))
display(rd)
top_role = rd["xg_p90"].idxmax(); top_int = rd["interceptions_p90"].idxmax()
display(Markdown(f"In this sample, **{top_role}s** have the highest median xG/90 ({rd.loc[top_role, 'xg_p90']}) and "
                 f"**{top_int}s** the highest median interceptions/90 ({rd.loc[top_int, 'interceptions_p90']}) — "
                 "which is why raw totals are never compared across roles."))
viz.correlation_heatmap(elig, eda_metrics, banner=BANNER).show()

**Observation (computed): median per-90 profile by role family**

,players,xg_p90,progressive_passes_p90,interceptions_p90,pass_completion_pct,pressures_p90
role_family,,,,,,
Defender,38,0.02,1.83,1.40,84.44,11.91
Forward,10,0.32,0.60,0.08,74.81,14.45
Goalkeeper,14,0.00,0.00,0.00,74.78,0.00
Midfielder,22,0.03,1.39,0.98,84.83,18.06


In this sample, **Forwards** have the highest median xG/90 (0.32) and **Defenders** the highest median interceptions/90 (1.4) — which is why raw totals are never compared across roles.

In [19]:
viz.results_overview(B.team_stats, FOCUS, banner=BANNER).show()
viz.team_trend(B.team_stats, FOCUS, banner=BANNER).show()

In [20]:
focus_ids = list(B.team_stats[B.team_stats.team_name == FOCUS].match_id)
av.dominance_scatter(B.team_stats, FOCUS, banner=BANNER).show()
av.phase_profile(B.flags, T["goals"], B.team_stats, FOCUS, cfg.match_phases, banner=BANNER).show()
av.xg_distribution(B.flags, FOCUS, focus_ids, banner=BANNER).show()
av.team_style_radar(B.team_stats, FOCUS, banner=BANNER).show()

In [21]:
# Match timeline and cumulative xG for the focus team's highest-xG match
fm = B.team_stats[B.team_stats.team_name == FOCUS].sort_values("xg", ascending=False).iloc[0]
mrow = T["matches"].set_index("match_id").loc[fm.match_id]
MATCH_LABEL = f"{mrow.match_date:%d %b %Y}: {mrow.home_team} {mrow.home_score}-{mrow.away_score} {mrow.away_team}"
viz.match_timeline(ta.match_timeline_events(T["events"], T["goals"], fm.match_id), title=f"Timeline — {MATCH_LABEL}",
                   banner=BANNER).show()
viz.cumulative_xg_chart(ta.cumulative_xg(B.flags, fm.match_id), title=f"Cumulative xG — {MATCH_LABEL}",
                        banner=BANNER).show()

In [22]:
match_fig = db.match_dashboard(B.flags, T["goals"], T["matches"], fm.match_id, banner=BANNER)
match_fig.show()
av.match_momentum(B.flags, T["goals"], fm.match_id, title=f"Momentum — {MATCH_LABEL}", banner=BANNER).show()

## 9 · Player intelligence centre
Role-aware leaderboards and comparisons (use the dropdowns inside each chart — they work in exported HTML too),
then a profile of one player against his benchmark group. Percentiles are relative to the eligible pool of *this
dataset*; with one club's season, opponents contribute only 1-2 matches each, so benchmarks for the focus squad are
partly internal (the radar title states the benchmark group and its size).

In [23]:
lb_metrics = ["xg_p90", "npxg_p90", "xa_p90", "key_passes_p90", "progressive_passes_p90", "progressive_carries_p90",
              "dribbles_completed_p90", "box_touches_p90", "pressures_p90", "interceptions_p90", "tackles_won_p90",
              "ball_recoveries_p90", "aerials_won_p90"]
viz.leaderboard_chart(B.season_pct, lb_metrics, min_minutes=max(cfg.min_minutes, 450), banner=BANNER).show()
viz.comparison_scatter(B.season_pct, lb_metrics + ["pass_completion_pct"], cfg.min_minutes, x="xg_p90",
                       y="xa_p90", highlight_team=FOCUS, banner=BANNER).show()

In [24]:
DEFAULT_PLAYER = "Florian Wirtz"
fs = B.season_pct[(B.season_pct.team_name == FOCUS)]
hit = fs[fs.player_name.str.contains(DEFAULT_PLAYER, case=False, na=False)]
P = (hit if len(hit) else fs[fs.primary_role != "GK"]).sort_values("minutes_played", ascending=False).iloc[0]

def player_view(player_id, team_id=None):
    prof = pa.player_profile(B.season_pct, player_id, team_id)
    display(pd.DataFrame([prof]))
    row = B.season_pct[B.season_pct.player_id == player_id].iloc[0]
    db.player_dashboard(B.season_pct, B.pms, B.flags, player_id, cfg, banner=BANNER).show()
    display(Markdown("**Observed relative strengths / weaknesses** (percentiles of role-fit metrics; not a rating)"))
    display(pa.strengths_weaknesses(B.season_pct, player_id, cfg, team_id))
    metrics = [m for m in pa.radar_metrics(row.primary_role, cfg) if m.endswith("_p90")][:3]
    viz.form_trend(pa.rolling_form_series(B.pms, player_id, metrics, cfg.form_window), metrics, row.player_name,
                   cfg.form_window, banner=BANNER).show()

player_view(P.player_id, P.team_id)

,player,team,primary_position,role,appearances,starts,minutes,goals,xG,assists,xA,coverage,benchmark
0,Florian Wirtz,Bayer Leverkusen,Left Attacking Midfield,Attacking midfielder,32,26,2529.0,11,8.47,10,7.73,high data coverage,"Midfielder (role family, n=22)"


**Observed relative strengths / weaknesses** (percentiles of role-fit metrics; not a rating)

,type,metric,value,percentile
0,relative strength,Prog. carries /90,4.98,98.0
1,relative strength,Completed dribbles /90,2.46,98.0
2,relative strength,Key passes /90,2.53,93.0
3,relative weakness,xA /90,0.28,93.0
4,relative weakness,Box touches /90,15.94,93.0
5,relative weakness,Shots /90,2.53,93.0


In [25]:
if WIDGETS_OK and not HEADLESS:
    opts = [(f"{r.player_name} ({r.primary_position}, {r.minutes_played:.0f} min)", r.player_id)
            for r in B.season_pct[B.season_pct.team_name == FOCUS].sort_values("minutes_played", ascending=False).itertuples()]
    W.interact(lambda player: player_view(player), player=W.Dropdown(options=opts, value=P.player_id,
                                                                     layout=W.Layout(width="480px")))
else:
    print("Live player selector available when running interactively (Colab/Jupyter).")

Live player selector available when running interactively (Colab/Jupyter).


In [26]:
# Head-to-head radar with the next most-used team-mate in the same role, plus spatial / directional profile
mates = fs[(fs.primary_role == P.primary_role) & (fs.player_id != P.player_id)].sort_values("minutes_played",
                                                                                          ascending=False)
viz.player_radar(B.season_pct, P.player_id, pa.radar_metrics(P.primary_role, cfg),
                 compare_id=mates.player_id.iloc[0] if len(mates) else None, banner=BANNER).show()
touch = B.flags[(B.flags.player_id == P.player_id) & B.flags.touches.astype(bool)]
av.action_heatmap(touch, f"{P.player_name}: where he receives and uses the ball (season)", banner=BANNER).show()
passes_p = B.flags[(B.flags.player_id == P.player_id) & (B.flags.type == "Pass")]
av.pass_direction_rose(passes_p, f"{P.player_name}: pass directions and completion", banner=BANNER).show()

### Squad-level views
The squad matrix shows every eligible outfield player's percentile on each metric (blue = high, red = low) — a
one-glance scouting sheet. The sunburst shows how playing time is spread across role families, roles and players
(click a segment to zoom).

In [27]:
squad_metrics = ["xg_p90", "xa_p90", "key_passes_p90", "progressive_passes_p90", "progressive_carries_p90",
                 "dribbles_completed_p90", "box_touches_p90", "pressures_p90", "interceptions_p90", "tackles_won_p90",
                 "ball_recoveries_p90", "aerials_won_p90", "pass_completion_pct"]
av.squad_percentile_heatmap(B.season_pct, FOCUS, squad_metrics, max(cfg.min_minutes, 450), banner=BANNER).show()
av.minutes_sunburst(B.season_pct, FOCUS, banner=BANNER).show()
viz.workload_bars(B.season_pct, FOCUS, banner=BANNER).show()

## 10 · Team and tactical intelligence
Formation use and outcomes (descriptive — opponent strength and squad availability confound them), team style,
and spatial views on a 120 × 80 pitch (attacking left → right for the team in question).

In [28]:
fsum = ta.formation_summary(B.team_stats, FOCUS)
display(fsum)
viz.formation_frequency(fsum, FOCUS, banner=BANNER).show()
viz.formation_comparison(fsum, FOCUS, banner=BANNER).show()

,starting_formation,matches,wins,draws,losses,points_per_match,xg_for,xg_against,goals_for,goals_against,possession_proxy_pct,field_tilt_pct,sample_warning
0,3-4-2-1,25,23,2,0,2.84,2.05,0.74,2.68,0.64,62.78,68.29,
1,3-4-3,8,4,4,0,2.00,2.44,1.10,2.50,0.88,58.40,66.92,
2,3-4-1-2,1,1,0,0,3.00,1.82,0.46,2.00,1.00,61.07,56.02,n<3: descriptive only


In [29]:
shots = B.flags[(B.flags.team_name == FOCUS) & (B.flags.type == "Shot")]
viz.shot_map(shots, f"{FOCUS}: all shots, season", banner=BANNER).show()
opp_shots = B.flags[(B.flags.match_id.isin(B.team_stats[B.team_stats.team_name == FOCUS].match_id))
                    & (B.flags.team_name != FOCUS) & (B.flags.type == "Shot")]
viz.shot_map(opp_shots, f"Opponents' shots against {FOCUS}", banner=BANNER).show()

In [30]:
pp = B.flags[(B.flags.match_id == fm.match_id) & (B.flags.player_id == P.player_id) & (B.flags.type == "Pass")]
viz.pass_map(pp, f"{P.player_name}: passes — {MATCH_LABEL}", banner=BANNER).show()
team_id_focus = B.team_stats[B.team_stats.team_name == FOCUS].team_id.iloc[0]
nodes, edges, cut = ta.pass_network(B.flags, T["appearances"], fm.match_id, team_id_focus)
viz.pass_network_chart(nodes, edges, f"{FOCUS} pass network — {MATCH_LABEL}", cut, banner=BANNER).show()
prog = B.flags[(B.flags.player_id == P.player_id)]
viz.progressive_actions_map(prog, f"{P.player_name}: progressive passes and carries (season)", banner=BANNER).show()

### Where the team presses and defends
Zone heatmaps of the focus team's pressures and ball-winning actions (tackles, interceptions, recoveries, blocks,
clearances) across the season, in the team's own attacking direction — pressing high shows up as heat in the
opponent half. The third map shows where opponents managed to have the ball against the focus team.

In [31]:
team_ev = B.flags[B.flags.team_id == team_id_focus]
av.action_heatmap(team_ev[team_ev.type == "Pressure"], f"{FOCUS}: pressures (season)", banner=BANNER).show()
av.action_heatmap(team_ev[team_ev.defensive_actions.astype(bool)], f"{FOCUS}: ball-winning and defensive actions",
                  banner=BANNER).show()
opp_ev = B.flags[B.flags.match_id.isin(focus_ids) & (B.flags.team_id != team_id_focus)]
av.action_heatmap(opp_ev[opp_ev.touches.astype(bool)], f"Opponents' on-ball actions against {FOCUS}",
                  banner=BANNER).show()
press_high = (team_ev[team_ev.type == "Pressure"].x >= 60).mean()
display(Markdown(f"**Observation (computed):** {press_high:.0%} of {FOCUS}'s pressures happen in the opponent half."))

**Observation (computed):** 57% of Bayer Leverkusen's pressures happen in the opponent half.

## 11 · Starting XI decision support
**Decision time:** before kick-off of a selected historical match. **Information used:** only matches *before*
that date. **Candidate pool:** that match's announced matchday squad.

**Score per (player, slot)** = Σ wᵢ·componentᵢ (+2 side-preference tie-breaker) × position compatibility, where the
components (0-100) are: role fit (weighted percentiles of the slot role's metrics), recent form (same composite over
the last 5 appearances; season value if none), sample reliability (minutes/900), workload suitability (penalises
>180 min in the previous 7 days or <3 rest days) and lineup continuity (started previous match).
**Optimiser:** the Hungarian algorithm assigns 11 distinct players to the 11 slots maximising the total — exact,
deterministic, and constrained (one goalkeeper, no duplicates, compatibility > 0, unavailable players excluded,
empty slot + warning when depth is insufficient). Scores are *not* probabilities.

In [32]:
focus_matches = B.team_stats[B.team_stats.team_name == FOCUS].sort_values("match_date").reset_index(drop=True)
XI_MATCH_INDEX = min(24, len(focus_matches) - 1)      # editable: which focus-team match to analyse
xm = focus_matches.iloc[XI_MATCH_INDEX]
XI_FORMATION = xm.starting_formation if xm.starting_formation in cfg.formation_templates else "4-3-3"
UNAVAILABLE = []                                      # e.g. [player_id, ...] marked unavailable by the user
ctx = lo.decision_context(B, FOCUS, match_id=xm.match_id)
xi = lo.optimize_xi(ctx, XI_FORMATION, cfg, unavailable=UNAVAILABLE)
print(f"Match: {xm.match_date:%d %b %Y} vs {xm.opponent} ({xm.home_away}) | formation used: {XI_FORMATION} "
      f"(actual {xm.starting_formation}) | candidates: {len(ctx.candidates)} | pool: {ctx.pool_description}")
viz.xi_pitch(xi.lineup, f"Recommended {XI_FORMATION} XI — decision date {ctx.asof:%Y-%m-%d} "
             f"(total score {xi.total_score:.0f})", banner=BANNER).show()
viz.score_decomposition(xi.lineup, cfg.xi_score_weights, "Explainable score decomposition per selected player",
                        banner=BANNER).show()
display(Markdown("**Warnings:** " + ("; ".join(xi.warnings) if xi.warnings else "none")))

Match: 10 Mar 2024 vs Wolfsburg (home) | formation used: 3-4-2-1 (actual 3-4-2-1) | candidates: 20 | pool: matchday squad of match 3895266 (2024-03-10)


**Warnings:** Thin depth at GK: 1 natural candidate(s).

In [33]:
cmp, agree = lo.compare_with_actual(xi, T["appearances"], xm.match_id, ctx.team_id)
display(Markdown(f"**Recommended vs actual XI** — descriptive agreement {agree:.0%} "
                 "(agreement is not proof of quality; disagreement is not proof the coach was wrong)."))
display(cmp)
display(Markdown("**Why each squad player was not selected**"))
display(xi.omitted)
display(Markdown("**Squad depth by slot** (candidates with compatibility ≥ 0.8)"))
display(xi.depth)

**Recommended vs actual XI** — descriptive agreement 91% (agreement is not proof of quality; disagreement is not proof the coach was wrong).

,player_name,in_recommended,in_actual
0,Edmond Fayçal Tapsoba,False,True
1,Exequiel Palacios,True,True
2,Florian Wirtz,True,True
3,Granit Xhaka,True,True
4,Jonas Hofmann,True,False
5,Jonathan Tah,True,True
6,Josip Stanišić,True,True
7,Lukáš Hrádecký,True,True
8,Nathan Tella,True,True
9,Patrik Schick,True,True


**Why each squad player was not selected**

,player_name,primary_role,minutes_before_decision,reason
0,Robert Andrich,DM,1060.0,Best slot LDM (score 69.0); Granit Xhaka score...
1,Borja Iglesias,CF,147.0,Best slot ST (score 60.0); Patrik Schick score...
2,Niklas Lomb,NaN,0.0,No minutes in analysed matches before decision...
3,Odilon Kossonou,CB,1406.0,Best slot RCB (score 67.0); Josip Stanišić sco...
4,Adam Hložek,AM,331.0,Best slot RAM (score 69.7); Jonas Hofmann scor...
5,Edmond Fayçal Tapsoba,CB,1493.0,Best slot LCB (score 72.2); Piero Hincapié sco...
6,Amine Adli,AM,600.0,Best slot LAM (score 74.4); Florian Wirtz scor...
7,Noah Mbamba,DM,23.0,Best slot RDM (score 65.7); Exequiel Palacios ...
8,Gustavo Adolfo Puerta Molano,DM,43.0,Best slot LDM (score 54.5); Granit Xhaka score...


**Squad depth by slot** (candidates with compatibility ≥ 0.8)

,slot,slot_role,candidates_compat_ge_0.8
0,GK,GK,1
1,RCB,CB,5
2,CB,CB,5
3,LCB,CB,5
4,RWB,WB,2
5,RDM,DM,5
6,LDM,DM,5
7,LWB,WB,2
8,RAM,AM,4
9,LAM,AM,4


In [34]:
display(Markdown("**Formation comparison** (same pool and date)"))
display(lo.compare_formations(ctx, cfg, UNAVAILABLE))
# Alternative "rotation" XI: rest the three selected outfield players with the most minutes in the previous 14 days
load = ctx.candidates.set_index("player_id")["minutes_prev_14d"].fillna(0)
rest = list(xi.lineup[xi.lineup.slot_role != "GK"].assign(l=lambda d: d.player_id.map(load))
            .nlargest(3, "l").player_id)
alt = lo.optimize_xi(ctx, XI_FORMATION, cfg, unavailable=UNAVAILABLE + rest)
viz.alternative_xi_comparison(xi.lineup, alt.lineup, "Recommended XI", "Rotation XI (3 highest-load rested)",
                              banner=BANNER).show()
print(f"Total score: recommended {xi.total_score:.1f} vs rotation {alt.total_score:.1f} "
      f"(Δ {alt.total_score - xi.total_score:+.1f})")

**Formation comparison** (same pool and date)

,formation,slots_filled,total_score,mean_slot_score,mean_compatibility,low_coverage_picks,warnings
0,3-4-2-1,11,843.7,76.7,1.00,0,1
1,3-4-3,11,797.3,72.5,0.94,0,1
2,3-5-2,11,782.1,71.1,0.96,1,2
3,4-2-3-1,11,778.3,70.8,0.94,0,1
4,4-3-3,11,766.3,69.7,0.91,0,1
5,4-1-4-1,11,735.3,66.8,0.88,0,1
6,4-4-2,11,718.5,65.3,0.88,1,2


Total score: recommended 843.7 vs rotation 753.7 (Δ -90.0)


In [35]:
# What-if: force the best-scoring unselected player into his best slot and re-optimise
cand_scores = xi.score_matrix[xi.score_matrix.feasible & ~xi.score_matrix.player_id.isin(xi.lineup.player_id)]
if len(cand_scores):
    best = cand_scores.sort_values("score", ascending=False).iloc[0]
    wi, delta = lo.what_if(ctx, xi, best.slot, best.player_id, cfg, UNAVAILABLE)
    display(Markdown(f"**What-if:** pin **{best.player_name}** at **{best.slot}** → total score change "
                     f"**{delta:+.1f}** (the optimiser's choice is optimal under the stated weights, so Δ ≤ 0)."))
    display(wi.lineup[["slot", "player_name", "score", "forced"]].round(1))

**What-if:** pin **Amine Adli** at **LAM** → total score change **-16.9** (the optimiser's choice is optimal under the stated weights, so Δ ≤ 0).

,slot,player_name,score,forced
0,GK,Lukáš Hrádecký,69.8,False
1,RCB,Josip Stanišić,72.0,False
2,CB,Jonathan Tah,73.0,False
3,LCB,Piero Hincapié,76.1,False
4,RWB,Nathan Tella,66.4,False
5,RDM,Exequiel Palacios,74.8,False
6,LDM,Granit Xhaka,69.4,False
7,LWB,Álex Grimaldo,85.1,False
8,RAM,Florian Wirtz,93.6,False
9,LAM,Amine Adli,74.4,True


In [36]:
if WIDGETS_OK and not HEADLESS:
    names = ctx.candidates.set_index("player_id")["player_name"].to_dict()
    def xi_live(formation, unavailable):
        r = lo.optimize_xi(ctx, formation, cfg, unavailable=list(unavailable))
        viz.xi_pitch(r.lineup, f"{formation} — total {r.total_score:.0f}", banner=BANNER).show()
        print("Warnings:", r.warnings or "none")
    W.interact(xi_live, formation=W.Dropdown(options=list(cfg.formation_templates), value=XI_FORMATION),
               unavailable=W.SelectMultiple(options=[(v, k) for k, v in names.items()], rows=8,
                                            description="Unavailable"))
else:
    print("Live formation / availability controls available when running interactively (Colab/Jupyter).")

Live formation / availability controls available when running interactively (Colab/Jupyter).


In [37]:
t0 = time.perf_counter()
hist_xi = lo.historical_agreement(B, FOCUS)
print(f"Re-ran the optimiser before each of {len(hist_xi)} matches using only prior data "
      f"({time.perf_counter() - t0:.1f}s). Mean descriptive agreement with the actual XI: "
      f"{hist_xi.agreement_pct.mean():.1f}% (median {hist_xi.agreement_pct.median():.0f}%).")
display(hist_xi.tail(8))
# Hypothetical next fixture (4 days after last analysed match)
ctx_next = lo.decision_context(B, FOCUS)
xi_next = lo.optimize_xi(ctx_next, XI_FORMATION, cfg)
viz.xi_pitch(xi_next.lineup, f"Hypothetical next-fixture XI ({XI_FORMATION}), decision date "
             f"{ctx_next.asof:%Y-%m-%d}", banner=BANNER).show()

Re-ran the optimiser before each of 33 matches using only prior data (7.1s). Mean descriptive agreement with the actual XI: 77.7% (median 82%).


,match_id,match_date,opponent,actual_formation,template_used,template_mapped,agreement_pct
25,3895286,2024-03-30,Hoffenheim,3-4-2-1,3-4-2-1,False,81.8
26,3895292,2024-04-06,Union Berlin,3-4-2-1,3-4-2-1,False,54.5
27,3895302,2024-04-14,Werder Bremen,3-4-2-1,3-4-2-1,False,72.7
28,3895309,2024-04-21,Borussia Dortmund,3-4-3,3-4-3,False,63.6
29,3895320,2024-04-27,VfB Stuttgart,3-4-2-1,3-4-2-1,False,63.6
30,3895333,2024-05-05,Eintracht Frankfurt,3-4-2-1,3-4-2-1,False,72.7
31,3895340,2024-05-12,Bochum,3-4-3,3-4-3,False,63.6
32,3895348,2024-05-18,Augsburg,3-4-2-1,3-4-2-1,False,72.7


## 12 · Substitution intelligence
Historical substitutions enriched with the **team-perspective score at the moment of the change**, timing band,
roles, window usage (rules user-configured) and manager. Then a live scenario: everything the bench could have known
at the decision minute — and nothing after it.

In [38]:
S = B.subs
display(S[["match_date", "team_name", "minute", "player_off_name", "off_role", "player_on_name", "on_role",
           "sub_reason", "game_state", "team_score", "opponent_score", "timing_band", "window_number",
           "manager_name"]].head(10))
print(f"{len(S)} substitutions in {S.match_id.nunique()} matches; identity check: "
      f"{(S.identity_check == 'ok').mean():.0%} resolved against matchday squads; "
      f"injury-coded: {(S.sub_reason == 'Injury').sum()}")
S_grp = S.assign(group=np.where(S.team_name == FOCUS, FOCUS, "Opponents (pooled)"))
viz.sub_timing_histogram(S_grp, cfg.sub_timing_bands, group="group", banner=BANNER).show()
viz.subs_by_state(S, banner=BANNER).show()
display(sa.state_profile(S_grp, by="group"))

,match_date,team_name,minute,player_off_name,off_role,player_on_name,on_role,sub_reason,game_state,team_score,opponent_score,timing_band,window_number,manager_name
0,2023-08-19,RB Leipzig,61,Xavi Simons,WM,Kevin Kampl,DM,Tactical,trailing,1,2,61-70,1,Marco Rose
1,2023-08-19,RB Leipzig,61,Nicolas Seiwald,DM,Junior Castello Lukeba,CB,Tactical,trailing,1,2,61-70,1,Marco Rose
2,2023-08-19,RB Leipzig,61,Timo Werner,WM,Yussuf Yurary Poulsen,CF,Tactical,trailing,1,2,61-70,1,Marco Rose
3,2023-08-19,Bayer Leverkusen,74,Florian Wirtz,W,Robert Andrich,DM,Tactical,leading,3,2,71-80,1,Xabi Alonso
4,2023-08-19,RB Leipzig,74,Ikoma Loïs Openda,CF,Emil Peter Forsberg,AM,Tactical,trailing,2,3,71-80,2,Marco Rose
5,2023-08-19,Bayer Leverkusen,77,Jeremie Frimpong,WB,Arthur Augusto de Matos Soares,WB,Injury,leading,3,2,71-80,2,Xabi Alonso
6,2023-08-19,RB Leipzig,83,Xaver Schlager,DM,Fabio Carvalho,DM,Tactical,trailing,2,3,81-90+,3,Marco Rose
7,2023-08-19,Bayer Leverkusen,88,Jonas Hofmann,CF,Adam Hložek,CF,Tactical,leading,3,2,81-90+,3,Xabi Alonso
8,2023-08-19,Bayer Leverkusen,88,Victor Okoh Boniface,CF,Nadiem Amiri,CF,Tactical,leading,3,2,81-90+,3,Xabi Alonso
9,2023-08-26,Borussia Mönchengladbach,66,Florian Neuhaus,DM,Rocco Reitz,DM,Tactical,trailing,0,3,61-70,1,Gerardo Seoane


301 substitutions in 34 matches; identity check: 100% resolved against matchday squads; injury-coded: 17


,group,game_state,substitutions,median_minute,mean_minute,attacking_change_pct
0,Bayer Leverkusen,drawing,16,67.5,68.2,0.0
1,Bayer Leverkusen,leading,112,79.0,77.0,3.6
2,Bayer Leverkusen,trailing,14,80.5,74.7,28.6
3,Opponents (pooled),drawing,35,74.0,67.8,0.0
4,Opponents (pooled),leading,13,73.0,74.1,15.4
5,Opponents (pooled),trailing,111,71.0,70.3,9.9


In [39]:
subs_fig = db.substitution_dashboard(S, FOCUS, cfg.sub_timing_bands, banner=BANNER)
subs_fig.show()
av.subs_team_band_heatmap(S, cfg.sub_timing_bands, banner=BANNER).show()
viz.role_flow_sankey(sa.role_flow(S), banner=BANNER).show()
viz.substitution_timeline(S[S.match_id == fm.match_id], f"Substitutions — {MATCH_LABEL}", banner=BANNER).show()

### Live scenario
The scenario is picked automatically as the first focus-team match in which the team was **not winning** at
~65 minutes of playing time (falls back to any match). Intent is inferred by a transparent rule — trailing → attack,
leading → protect, drawing → balance — and can be overridden. Outgoing candidates are judged on minutes played *in
this match*, booking risk, and in-match role actions relative to **their own pre-match per-90 baseline** (role
context, never a raw low count). Incoming candidates are bench players who have not yet appeared.

In [40]:
SCENARIO_MINUTE = 65.0
INTENT_OVERRIDE = None          # "attack" | "protect" | "balance" | None (rule-based)
scen = None
for r in focus_matches.itertuples():
    opp_id = r.opponent_id
    tg, og = features.score_at(T["goals"], r.match_id, r.team_id, opp_id, elapsed=SCENARIO_MINUTE)
    if tg <= og:
        scen = r; break
scen = scen or focus_matches.iloc[0]
rec = sa.recommend_substitution(B, scen.match_id, FOCUS, SCENARIO_MINUTE, intent=INTENT_OVERRIDE)
c = rec["context"]
display(Markdown(f"**Scenario:** {scen.match_date:%d %b %Y}, {FOCUS} vs {c['opponent']} ({c['home_away']}) at "
                 f"**{SCENARIO_MINUTE:.0f}'** of playing time — score **{c['team_score']}-{c['opponent_score']}** "
                 f"({c['game_state']}), numerical state {c['man_advantage']:+d}, substitutions remaining "
                 f"**{c['subs_remaining']}**, windows remaining **{c['windows_remaining']}** · rules: {c['rule_source']}"))
display(pd.DataFrame([c["team_live"], c["opponent_live"]], index=[FOCUS, c["opponent"]]))
display(Markdown(f"**Intent:** `{rec['intent']}` — {rec['intent_reason']}"))
display(Markdown("**Outgoing-candidate ranking** (data up to the decision minute + pre-match baselines)"))
display(rec["outgoing_ranking"][["player_name", "role", "minutes_so_far", "booked", "involvement_vs_baseline",
                                 "outgoing_score", "reasons"]])

**Scenario:** 15 Sep 2023, Bayer Leverkusen vs Bayern Munich (away) at **65'** of playing time — score **1-1** (drawing), numerical state +0, substitutions remaining **4**, windows remaining **3** · rules: User-configured default (IFAB Law 3 five-substitute option). Verify against the competition's regulations.

,shots,xg,passes,pass_completion_pct,final_third_touches,pressures,defensive_actions
Bayer Leverkusen,7,0.42,393,85.0,240,109,93
Bayern Munich,14,1.88,474,85.9,332,124,92


**Intent:** `balance` — Drawing: balance control, chance creation and defensive risk.

**Outgoing-candidate ranking** (data up to the decision minute + pre-match baselines)

,player_name,role,minutes_so_far,booked,involvement_vs_baseline,outgoing_score,reasons
0,Florian Wirtz,W,65.0,True,0.51,55.0,65 min played this match; role-action rate at ...
1,Jonas Hofmann,W,65.0,True,0.61,51.9,65 min played this match; role-action rate at ...
2,Alejandro Grimaldo García,WB,65.0,False,0.70,34.2,65 min played this match; role-action rate at ...
3,Jonathan Tah,CB,65.0,False,0.78,31.7,65 min played this match; role-action rate at ...
4,Victor Okoh Boniface,CF,65.0,False,0.82,30.8,65 min played this match; role-action rate at ...
5,Jeremie Frimpong,WB,65.0,False,0.88,28.9,65 min played this match; role-action rate at ...
6,Granit Xhaka,DM,65.0,False,0.91,28.0,65 min played this match; role-action rate at ...
7,Edmond Fayçal Tapsoba,CB,65.0,False,1.12,25.3,65 min played this match; role-action rate at ...
8,Odilon Kossonou,CB,65.0,False,1.15,25.3,65 min played this match; role-action rate at ...
9,Exequiel Alejandro Palacios,DM,18.0,False,0.78,13.5,18 min played this match; role-action rate at ...


In [41]:
if rec["status"] == "ok":
    off, on = rec["recommended_off"], rec["recommended_on"]
    inc = sa.rank_incoming(c, off["role"], rec["intent"], cfg)
    viz.candidate_comparison(inc, cfg.sub_candidate_weights, f"Incoming candidates for {off['player_name']} "
                             f"({off['role']}) — intent '{rec['intent']}'", banner=BANNER).show()
    card = pd.DataFrame({"field": ["Recommended OFF", "Recommended ON", "Suggested decision window",
                                   "Tactical objective", "Evidence (incoming)", "Evidence (outgoing)",
                                   "Risks / trade-offs", "Rule allowance", "Data coverage", "Injury status",
                                   "Final decision"],
                         "value": [f"{off['player_name']} ({off['role']})", f"{on['player_name']} ({on['role']})",
                                   rec["decision_window"], rec["tactical_objective"], on["evidence"], off["reasons"],
                                   on["risks"], rec["rule_status"], rec["coverage"], on["injury_status"],
                                   rec["human_note"]]})
    display(card.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))
    display(Markdown("**Alternatives**"))
    display(rec["alternatives"])
else:
    print("Recommendation status:", rec["status"])

field,value
Recommended OFF,Florian Wirtz (W)
Recommended ON,Adam Hložek (W)
Suggested decision window,now to ~75' of playing time; team's historical drawing substitutions mostly fall between 58' and 83'
Tactical objective,maintain control while adding a fresh option
Evidence (incoming),role W vs outgoing W: compatibility 1.00; 'balance' metric fit 45th pct; 34 pre-match minutes; recent form 50
Evidence (outgoing),65 min played this match; role-action rate at 51% of own pre-match baseline (6 role metrics); booked (second-yellow risk)
Risks / trade-offs,small evidence base
Rule allowance,4 substitutions and 3 windows remaining (User-configured default (IFAB Law 3 five-substitute option). Verify against the competition's regulations.)
Data coverage,low data coverage for the incoming player
Injury status,unknown (no injury data in source — not assumed fit)


**Alternatives**

,player_name,role,candidate_score,evidence,risks
1,Nathan Tella,WB,50.4,role WB vs outgoing W: compatibility 0.60; 'ba...,small evidence base
2,Gustavo Adolfo Puerta Molano,AM,41.9,role AM vs outgoing W: compatibility 0.80; 'ba...,small evidence base
3,Nadiem Amiri,CF,37.5,role CF vs outgoing W: compatibility 0.60; 'ba...,small evidence base


In [42]:
if WIDGETS_OK and not HEADLESS:
    mopts = [(f"{r.match_date:%d %b} vs {r.opponent}", r.match_id) for r in focus_matches.itertuples()]
    def sub_live(match_id, minute, intent):
        r = sa.recommend_substitution(B, match_id, FOCUS, float(minute), intent=None if intent == "auto" else intent)
        cc = r["context"]
        print(f"{cc['team_score']}-{cc['opponent_score']} ({cc['game_state']}) at {minute}' | intent {r['intent']} | "
              f"status {r['status']}")
        if r["status"] == "ok":
            print(f"OFF {r['recommended_off']['player_name']} → ON {r['recommended_on']['player_name']}")
            print("Evidence:", r["recommended_on"]["evidence"]); print("Risks:", r["recommended_on"]["risks"])
    W.interact(sub_live, match_id=W.Dropdown(options=mopts, value=scen.match_id, description="Match"),
               minute=W.IntSlider(value=65, min=46, max=88, step=1), intent=["auto", "attack", "protect", "balance"])
else:
    print("Live substitution scenario controls available when running interactively (Colab/Jupyter).")

Live substitution scenario controls available when running interactively (Colab/Jupyter).


In [43]:
t0 = time.perf_counter()
agree_subs = sa.historical_agreement(B, FOCUS, top_k=3)
print(f"Heuristic re-run 1 second before each of {len(agree_subs)} non-injury {FOCUS} substitutions "
      f"({time.perf_counter() - t0:.1f}s), pre-decision data only.")
summary = pd.DataFrame({
    "measure": ["actual incoming player in heuristic top-3", "actual outgoing player in heuristic top-3",
                "median bench size", "random top-3 baseline for incoming (3 / bench size, mean)"],
    "value": [f"{agree_subs.on_in_top_k.mean():.0%}", f"{agree_subs.off_in_top_k.mean():.0%}",
              f"{agree_subs.bench_size.median():.0f}", f"{(3 / agree_subs.bench_size.clip(lower=3)).mean():.0%}"]})
display(summary)
display(Markdown("Agreement describes how often the transparent heuristic points where the coach went; it does not "
                 "measure decision quality (selection bias: we only observe the chosen option's outcome)."))

Heuristic re-run 1 second before each of 137 non-injury Bayer Leverkusen substitutions (25.1s), pre-decision data only.


,measure,value
0,actual incoming player in heuristic top-3,82%
1,actual outgoing player in heuristic top-3,57%
2,median bench size,7
3,random top-3 baseline for incoming (3 / bench ...,43%


Agreement describes how often the transparent heuristic points where the coach went; it does not measure decision quality (selection bias: we only observe the chosen option's outcome).

In [44]:
imp = sa.impact_windows(B)
display(Markdown(f"**Pre/post windows** (±{cfg.impact_window_minutes:.0f} min, rates per 15 min). Unit = a team's "
                 f"substitution *decision* (simultaneous changes grouped): {len(imp)} decisions with ≥3 min on both "
                 f"sides. {imp.contaminated.mean():.0%} contain a goal, red card or another own decision "
                 f"(goal {imp.goal_in_window.mean():.0%}, own other decision {imp.other_own_decision_in_window.mean():.0%}); "
                 f"the opponent also changed in {imp.opponent_changed_in_window.mean():.0%}."))
display(Markdown("All windows:")); display(sa.impact_summary(imp))
display(Markdown("Clean windows only:")); display(sa.impact_summary(imp, clean_only=True))
viz.impact_chart(imp, "xg", banner=BANNER).show()

**Pre/post windows** (±10 min, rates per 15 min). Unit = a team's substitution *decision* (simultaneous changes grouped): 189 decisions with ≥3 min on both sides. 71% contain a goal, red card or another own decision (goal 49%, own other decision 45%); the opponent also changed in 79%.

All windows:

,game_state,n_decisions,opponent_also_changed_pct,delta_team_shots,delta_team_xg,delta_team_final_third_touches,delta_team_progressive_passes,delta_team_pressures
0,drawing,38,61.0,0.0,-0.011,-4.500,-1.5,1.500
1,leading,76,89.0,0.0,-0.007,1.500,0.0,1.952
2,trailing,75,79.0,0.0,0.000,13.315,1.5,-1.500


Clean windows only:

,game_state,n_decisions,opponent_also_changed_pct,delta_team_shots,delta_team_xg,delta_team_final_third_touches,delta_team_progressive_passes,delta_team_pressures
0,drawing,13,69.0,0.000,0.000,6.000,-1.5,0.00
1,leading,16,81.0,0.246,0.041,11.404,1.5,0.75
2,trailing,25,76.0,0.000,0.000,13.315,1.5,3.00


> **Causal-inference rule.** These are *observed associations*. Teams that are trailing push forward anyway
> (score effects), opponents change too, fatigue accumulates, time remaining differs and extreme pre-windows regress
> to the mean. Nothing here shows that a substitution *caused* a change. A defensible causal design would need, e.g.,
> matched comparison windows without substitutions in the same game state, which is outside this project's scope.

## 13 · Managerial decision profile
Measured: **observed tactical flexibility** (formation diversity, in-match shifts), **rotation tendency** (lineup
changes), **substitution timing profile** and **game-state response profile**. Not measured — because no valid
instrument exists in event data — leadership, motivation, man-management or "managerial skill". Manager identity
comes from the source's match metadata; managers with fewer than `min_manager_matches` matches are shown as
descriptive only and are never ranked.

In [45]:
display(Markdown(f"**Manager identity source:** {B.manager_source} · coverage "
                 f"{T['managers'].match_id.nunique()}/{len(T['matches'])} matches"))
prof = ma.manager_profiles(B)
display(prof)
FOCUS_MANAGER = B.team_stats[B.team_stats.team_name == FOCUS].manager_name.mode().iloc[0] \
    if B.team_stats.manager_name.notna().any() else None
if FOCUS_MANAGER:
    display(Markdown(f"**{FOCUS_MANAGER} vs pooled small-sample managers** — a reference, not a ranking "
                     "(all pooled managers faced the same opponent, so context differs)."))
    display(ma.pooled_reference(prof, FOCUS_MANAGER, cfg))

**Manager identity source:** StatsBomb matches.json (managers field) · coverage 34/34 matches

,manager,team,matches,sample_status,points_per_match,xg_diff_per_match,formations_used,most_used_formation,most_used_share_pct,formation_diversity_index,in_match_formation_changes_per_match,mean_lineup_changes,continuity_pairs,subs_per_match,median_sub_minute,median_first_sub_minute,early_subs_pct_before_60,late_subs_pct_75_plus,multi_sub_window_pct,subs_when_trailing_pct,subs_when_leading_pct,like_for_like_pct,injury_subs,substitutions
0,Xabi Alonso,Bayer Leverkusen,34,comparable sample,2.65,1.32,3,3-4-2-1,73.5,0.61,0.5,2.21,33,4.18,79.0,63.5,7.7,61.3,61.3,9.9,78.9,76.8,5,142
1,Jess Christian Thorup,Augsburg,2,descriptive only (n<10),0.00,-1.32,1,4-1-2-1-2,100.0,0.00,0.5,5.00,1,5.00,79.0,50.5,10.0,60.0,70.0,50.0,0.0,80.0,1,10
2,Torsten Lieberknecht,Darmstadt 98,2,descriptive only (n<10),0.00,-1.95,2,3-5-2,50.0,1.00,1.0,4.00,1,5.00,66.0,44.5,30.0,20.0,90.0,90.0,0.0,20.0,4,10
3,Thomas Tuchel,Bayern Munich,2,descriptive only (n<10),0.50,-0.21,2,4-2-3-1,50.0,1.00,1.5,6.00,1,5.00,69.5,59.5,20.0,40.0,80.0,50.0,0.0,60.0,0,10
4,Sebastian Hoeneß,VfB Stuttgart,2,descriptive only (n<10),1.00,0.51,2,3-4-2-1,50.0,1.00,1.5,3.00,1,5.00,78.0,69.0,0.0,70.0,80.0,0.0,50.0,60.0,1,10
5,Pellegrino Matarazzo,Hoffenheim,2,descriptive only (n<10),0.00,-0.96,1,3-5-2,100.0,0.00,0.0,6.00,1,4.00,77.0,66.5,12.5,62.5,75.0,50.0,50.0,87.5,0,8
6,Ole Werner,Werder Bremen,2,descriptive only (n<10),0.00,-1.61,1,3-5-2,100.0,0.00,0.5,5.00,1,4.00,71.0,61.5,12.5,37.5,75.0,100.0,0.0,87.5,1,8
7,Marco Rose,RB Leipzig,2,descriptive only (n<10),0.00,-0.30,2,4-4-2,50.0,1.00,0.5,3.00,1,5.00,74.0,67.0,0.0,20.0,70.0,50.0,0.0,60.0,0,10
8,Niko Kovač,Wolfsburg,2,descriptive only (n<10),0.00,-1.16,2,3-5-2,50.0,1.00,2.0,4.00,1,4.50,70.0,38.0,22.2,44.4,66.7,77.8,0.0,55.6,0,9
9,Gerardo Seoane,Borussia Mönchengladbach,2,descriptive only (n<10),0.50,-2.15,2,3-4-3,50.0,1.00,0.5,6.00,1,5.00,78.0,63.5,0.0,60.0,70.0,50.0,0.0,70.0,0,10


**Xabi Alonso vs pooled small-sample managers** — a reference, not a ranking (all pooled managers faced the same opponent, so context differs).

,indicator,Xabi Alonso,"pooled other managers (n=34 matches, 21 managers)"
0,subs_per_match,4.18,4.68
1,median_sub_minute,79.00,70.96
2,median_first_sub_minute,63.50,55.79
3,early_subs_pct_before_60,7.70,12.43
4,late_subs_pct_75_plus,61.30,41.94
5,multi_sub_window_pct,61.30,69.81
6,formation_diversity_index,0.61,0.59
7,in_match_formation_changes_per_match,0.50,0.82


In [46]:
if FOCUS_MANAGER:
    viz.manager_formation_distribution(B.team_stats, FOCUS_MANAGER, banner=BANNER).show()
    cont = ma.lineup_continuity(T["appearances"], T["matches"])
    viz.rotation_trend(cont[cont.team_id == team_id_focus], FOCUS, banner=BANNER).show()
    rotation_fig = av.rotation_matrix(T["appearances"], T["matches"], FOCUS, banner=BANNER)
    rotation_fig.show()
    av.lineup_similarity(T["appearances"], T["matches"], FOCUS, banner=BANNER).show()
    viz.manager_sub_profile(S, FOCUS_MANAGER, cfg.sub_timing_bands, banner=BANNER).show()
    resp = ma.response_after_conceding(B)
    r = resp[resp.manager == FOCUS_MANAGER]
    display(Markdown(f"**Response after conceding ({FOCUS_MANAGER}):** {len(r)} goals conceded; median "
                     f"{r.minutes_to_next_sub.median():.1f} min to the next substitution where one followed "
                     f"({r.minutes_to_next_sub.notna().sum()} cases) — confounded by when goals happen."))

**Response after conceding (Xabi Alonso):** 24 goals conceded; median 24.0 min to the next substitution where one followed (24 cases) — confounded by when goals happen.

## 14 · Optional modelling (data-gated): who starts?
| | |
|---|---|
| Prediction unit | player in the focus team's matchday squad, per match |
| Target | named in the Starting XI (1/0) |
| Prediction time | before kick-off |
| Features | only from earlier matches: started previous match, consecutive starts, start share, minutes share, minutes in previous 7/14 days, rest days, goalkeeper flag |
| Split | chronological by match date (first 70 % of matches train, rest test) — never random |
| Baseline | "started the previous match" |
| Gate | ≥ 300 rows and ≥ 15 matches, otherwise no model is fitted |

This models what the coaching staff *did* (observed selection behaviour), not what was optimal.

In [47]:
model = mdl.fit_start_model(B, FOCUS)
display(pd.DataFrame([model["gate"]]))
if model["fitted"]:
    print(model["split"], f"| train rows {model['train_rows']} | test rows {model['test_rows']}")
    display(model["metrics"])
    display(model["coefficients"])
    display(Markdown("**Leakage audit**")); display(model["leakage_audit"])
    viz.confusion_matrix_chart(model["confusion_matrix"], banner=BANNER).show()
    viz.roc_chart(model["roc"], float(model["metrics"].roc_auc.dropna().iloc[0]), banner=BANNER).show()
    viz.calibration_chart(model["calibration"], banner=BANNER).show()
else:
    display(Markdown(f"**No model fitted:** {model['reason']} More matches for one team (e.g. a full season with "
                     "cup fixtures) would be required."))

,rows,matches,min_rows,min_matches,class_balance_starter_pct
0,660,33,300,15,55.0


chronological: train up to 2024-03-03, test from 2024-03-10 | train rows 460 | test rows 200


,model,precision,recall,f1,roc_auc,brier
0,Baseline: started previous match,0.743,0.682,0.711,NaN,NaN
1,Logistic regression (top-11 per match),0.745,0.745,0.745,0.775,0.197


,feature,coefficient_std
3,minutes_share_before,1.357
7,is_gk,0.924
0,started_prev_match,0.525
1,consecutive_starts_before,-0.323
2,start_share_before,0.321
5,minutes_prev_14d,0.259
4,minutes_prev_7d,0.143
6,rest_days_filled,-0.123


**Leakage audit**

,feature,computed from,availability
0,started_prev_match,team's previous match lineup,known before kick-off
1,consecutive_starts_before,previous matches,known before kick-off
2,start_share_before,starts / team matches before this match,known before kick-off
3,minutes_share_before,minutes / (team matches x 95) before this match,known before kick-off
4,minutes_prev_7d / 14d,minutes in earlier matches within 7/14 days,known before kick-off
5,rest_days_filled,days since last appearance (60 if none),known before kick-off
6,is_gk,role from squad match position (event-derived),position played in this match — mild contempor...
7,"EXCLUDED: minutes_played, events, goals in the...",same-match outcomes,would leak — not used


## 15 · Findings and research interpretation
Answers are generated from the executed data (re-running with another competition updates them). They are
descriptive statements about this sample, with the main alternative explanation alongside.

In [48]:
fam = rd
fs_ = ta.formation_summary(B.team_stats, FOCUS)
fsub = S[S.team_name == FOCUS]
osub = S[S.team_name != FOCUS]
cont_f = ma.lineup_continuity(T["appearances"], T["matches"]).query("team_id == @team_id_focus")
rows = [
 ("RQ1 Role-distinguishing attributes",
  f"Median xG/90 peaks for {fam['xg_p90'].idxmax()}s ({fam['xg_p90'].max()}), interceptions/90 for "
  f"{fam['interceptions_p90'].idxmax()}s, pass completion for {fam['pass_completion_pct'].idxmax()}s "
  f"({fam['pass_completion_pct'].max()}%).", "Role labels are event-position based; tactical roles vary within labels."),
 ("RQ2 Per-90 and role normalisation",
  f"{int(B.season.eligible.sum())} of {len(B.season)} players pass {cfg.min_minutes:.0f} min; raising the threshold "
  f"to 900 leaves {(B.season.minutes_played >= 900).sum()} — leaderboards change (sensitivity table, §7).",
  "Small minutes inflate per-90 extremes."),
 ("RQ3 Formations and outcomes",
  f"{FOCUS} used {len(fs_)} starting formations; most used {fs_.iloc[0]['starting_formation']} "
  f"({fs_.iloc[0]['matches']} matches, {fs_.iloc[0]['points_per_match']} ppm, xG diff "
  f"{fs_.iloc[0]['xg_for'] - fs_.iloc[0]['xg_against']:+.2f}/match).",
  "Formation choice depends on opponent and availability — not a causal comparison."),
 ("RQ4 Substitution timing by state",
  f"{FOCUS}: median sub minute {fsub.minute.median():.0f}', {(fsub.game_state == 'leading').mean():.0%} made while "
  f"leading. Opponents: median {osub.minute.median():.0f}', {(osub.game_state == 'trailing').mean():.0%} while trailing.",
  "Score state is itself driven by team strength (the focus team led most matches)."),
 ("RQ5 Substitute matching",
  f"Heuristic top-3 contained the actual incoming player in {agree_subs.on_in_top_k.mean():.0%} of non-injury subs "
  f"(random ≈ {(3 / agree_subs.bench_size.clip(lower=3)).mean():.0%}).", "Agreement ≠ quality; bench composition varies."),
 ("RQ6 Rotation, minutes and rest",
  f"Mean {cont_f.lineup_changes.mean():.1f} starting-XI changes between consecutive dataset matches; "
  f"pre-match start model F1 {model['metrics'].f1.iloc[1]:.2f} vs 'started previous match' baseline "
  f"{model['metrics'].f1.iloc[0]:.2f} (ROC-AUC {model['metrics'].roc_auc.dropna().iloc[0]:.2f})." if model.get("fitted") else
  f"Mean {cont_f.lineup_changes.mean():.1f} lineup changes; model gate not met.",
  "Cup/European fixtures are missing, so rest days are over-estimated."),
 ("RQ7 Observable manager patterns",
  f"{FOCUS_MANAGER}: {prof.iloc[0]['subs_per_match']} subs/match, first sub median "
  f"{prof.iloc[0]['median_first_sub_minute']:.0f}', formation diversity {prof.iloc[0]['formation_diversity_index']}."
  if FOCUS_MANAGER else "Manager identity unavailable.", "Other managers have ≤2 matches — descriptive only."),
 ("RQ8 Limits to causal claims",
  f"{imp.contaminated.mean():.0%} of pre/post substitution-decision windows contain a goal, red card or another "
  f"own decision, and the opponent also changed in {imp.opponent_changed_in_window.mean():.0%}; deltas are "
  "associations only.", "Score effects, opponent changes, fatigue, regression to the mean."),
]
findings = pd.DataFrame(rows, columns=["research question", "observed answer (this dataset)", "main caveat"])
display(findings.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))

research question,observed answer (this dataset),main caveat
RQ1 Role-distinguishing attributes,"Median xG/90 peaks for Forwards (0.32), interceptions/90 for Defenders, pass completion for Midfielders (84.83%).",Role labels are event-position based; tactical roles vary within labels.
RQ2 Per-90 and role normalisation,"84 of 377 players pass 180 min; raising the threshold to 900 leaves 17 — leaderboards change (sensitivity table, §7).",Small minutes inflate per-90 extremes.
RQ3 Formations and outcomes,"Bayer Leverkusen used 3 starting formations; most used 3-4-2-1 (25 matches, 2.84 ppm, xG diff +1.31/match).",Formation choice depends on opponent and availability — not a causal comparison.
RQ4 Substitution timing by state,"Bayer Leverkusen: median sub minute 79', 79% made while leading. Opponents: median 72', 70% while trailing.",Score state is itself driven by team strength (the focus team led most matches).
RQ5 Substitute matching,Heuristic top-3 contained the actual incoming player in 82% of non-injury subs (random ≈ 43%).,Agreement ≠ quality; bench composition varies.
"RQ6 Rotation, minutes and rest",Mean 2.2 starting-XI changes between consecutive dataset matches; pre-match start model F1 0.74 vs 'started previous match' baseline 0.71 (ROC-AUC 0.78).,"Cup/European fixtures are missing, so rest days are over-estimated."
RQ7 Observable manager patterns,"Xabi Alonso: 4.18 subs/match, first sub median 64', formation diversity 0.61.",Other managers have ≤2 matches — descriptive only.
RQ8 Limits to causal claims,"71% of pre/post substitution-decision windows contain a goal, red card or another own decision, and the opponent also changed in 79%; deltas are associations only.","Score effects, opponent changes, fatigue, regression to the mean."


**Recommendations for further data collection.** (1) All competitions a team plays (cups, Europe) for valid
workload/rotation proxies; (2) injury/availability feeds from a licensed provider; (3) several full league seasons
(e.g. StatsBomb's complete 2015/16 leagues) for role benchmarks independent of one club; (4) tracking or 360 data
for off-ball and pressing metrics; (5) a manager-tenure table to separate team and manager effects.

## 16 · Export and reproducibility
Writes processed tables (`data/processed/`), the run configuration and source manifest (`outputs/exports/`), the
auto-generated **dataset card** and **data dictionary** (`reports/`), a results summary, and interactive HTML copies
of key charts (`outputs/figures/`, PNG too if `kaleido` is installed). In `DEMO_SYNTHETIC` mode everything goes to
`synthetic/` sub-folders and the cards to `outputs/exports/synthetic/`, so synthetic results never overwrite real ones.

In [49]:
inv_out = reporting.export_tables(B)
display(inv_out)
paths = [reporting.write_run_config(B, {"focus_team": FOCUS, "xi_match_id": int(xm.match_id),
                                       "scenario_match_id": int(scen.match_id)}),
         reporting.write_manifest(B),
         reporting.write_dataset_card(B, cfg.reports_dir / "DATASET_CARD.md"),
         reporting.write_data_dictionary(B, cfg.reports_dir / "DATA_DICTIONARY.md"),
         reporting.write_results_summary(B, {"Dataset scale": scale, "Quality checks": B.quality,
                                             "Findings": findings, "Recommended XI": xi.lineup[
                                                 ["slot", "player_name", "score", "coverage_label"]].round(1),
                                             "Manager profiles": prof})]
for p in paths:
    print("written:", Path(p).relative_to(ROOT))

,table,rows,columns,csv,parquet
0,matches,34,23,matches.csv,matches.parquet
1,managers,68,8,managers.csv,managers.parquet
2,appearances,1353,26,appearances.csv,appearances.parquet
3,substitutions,301,41,substitutions.csv,substitutions.parquet
4,goals,113,8,goals.csv,goals.parquet
5,formations,195,10,formations.csv,formations.parquet
6,player_match_stats,1049,77,player_match_stats.csv,player_match_stats.parquet
7,player_season,377,134,player_season.csv,player_season.parquet
8,team_match_stats,68,47,team_match_stats.csv,team_match_stats.parquet
9,workload,1353,15,workload.csv,workload.parquet


written: outputs\exports\run_config.json
written: outputs\exports\source_manifest.csv
written: reports\DATASET_CARD.md
written: reports\DATA_DICTIONARY.md
written: outputs\exports\results_summary.md


In [50]:
export_figs = {
    "01_coverage_cards": viz.coverage_cards(scale, banner=BANNER),
    "04_leaderboard": viz.leaderboard_chart(B.season_pct, lb_metrics, max(cfg.min_minutes, 450), banner=BANNER),
    "05_player_radar": viz.player_radar(B.season_pct, P.player_id, pa.radar_metrics(P.primary_role, cfg), banner=BANNER),
    "06_comparison_scatter": viz.comparison_scatter(B.season_pct, lb_metrics, cfg.min_minutes, x="xg_p90",
                                                    y="xa_p90", highlight_team=FOCUS, banner=BANNER),
    "13_xi_pitch": viz.xi_pitch(xi.lineup, f"Recommended {XI_FORMATION} XI ({ctx.asof:%Y-%m-%d})", banner=BANNER),
    "15_shot_map": viz.shot_map(shots, f"{FOCUS}: shots", banner=BANNER),
    "17_pass_network": viz.pass_network_chart(nodes, edges, f"{FOCUS} pass network", cut, banner=BANNER),
    "20_sub_timing": viz.sub_timing_histogram(S_grp, cfg.sub_timing_bands, group="group", banner=BANNER),
    "22_role_flow": viz.role_flow_sankey(sa.role_flow(S), banner=BANNER),
    "23_impact": viz.impact_chart(imp, "xg", banner=BANNER),
    "31_season_dashboard": season_fig,
    "32_player_dashboard": db.player_dashboard(B.season_pct, B.pms, B.flags, P.player_id, cfg, banner=BANNER),
    "33_match_dashboard": match_fig,
    "34_substitution_dashboard": subs_fig,
    "35_rotation_matrix": rotation_fig,
    "36_squad_percentiles": av.squad_percentile_heatmap(B.season_pct, FOCUS, squad_metrics,
                                                        max(cfg.min_minutes, 450), banner=BANNER),
    "37_pressure_heatmap": av.action_heatmap(team_ev[team_ev.type == "Pressure"], f"{FOCUS}: pressures",
                                             banner=BANNER),
}
for name, fig in export_figs.items():
    res = viz.export_figure(fig, str(cfg.figures_dir / f"{name}.html"), str(cfg.figures_dir / f"{name}.png"))
print(f"{len(export_figs)} interactive HTML charts written to {cfg.figures_dir.relative_to(ROOT)} "
      f"(PNG: {res.get('png')})")

# One-page presentation dashboard (open in any browser; Plotly JS from CDN)
dash_sections = [
    ("Season overview", "KPIs, points vs shot-based expected points, xG for/against per match.", season_fig),
    ("Player card", f"{P.player_name}: percentile profile, action heatmap and match-by-match output.",
     export_figs["32_player_dashboard"]),
    ("Squad profile", "Percentile of every eligible outfield player on each metric.", export_figs["36_squad_percentiles"]),
    ("Who played when", "Minutes per player per match: rotation across the season.", rotation_fig),
    ("Recommended starting XI", f"Pre-match recommendation for {xm.match_date:%d %b %Y} vs {xm.opponent} "
     f"(decision support only).", export_figs["13_xi_pitch"]),
    ("Match dashboard", MATCH_LABEL, match_fig),
    ("Pressing map", "Where the team applies pressure.", export_figs["37_pressure_heatmap"]),
    ("Substitution patterns", "Timing, game state and most-used substitutes.", subs_fig),
    ("Substitution windows", "Observed association only — not a causal effect.", export_figs["23_impact"]),
]
dash_path = ROOT / "outputs" / ("dashboard_synthetic.html" if cfg.is_synthetic else "dashboard.html")
reporting.write_dashboard_html(dash_sections, dash_path,
                               title=f"{FOCUS} — Football Decision Intelligence Dashboard",
                               subtitle=f"{T['matches'].competition_name.iloc[0]} {T['matches'].season_name.iloc[0]} · "
                                        f"{len(T['matches'])} matches · {len(T['events']):,} events · StatsBomb "
                                        "open data" if not cfg.is_synthetic else "Synthetic demo league",
                               banner=BANNER)
print("presentation dashboard:", dash_path.relative_to(ROOT))
if IN_COLAB:
    import shutil
    from google.colab import files as _f
    shutil.make_archive("/content/football_outputs", "zip", ROOT / "outputs")
    _f.download("/content/football_outputs.zip")

17 interactive HTML charts written to outputs\figures (PNG: skipped (ValueError))
presentation dashboard: outputs\dashboard.html


## 17 · Conclusion and viva preparation
**Plain-language summary.** The notebook turns StatsBomb event and lineup files into audited tables, role-aware
player metrics and interactive views, and uses them for two decisions coaches actually make — who starts and who comes
on — with every number traceable to a definition and every recommendation limited to information available at the
time.

| The system **can** | The system **cannot** |
|---|---|
| Rank squad players for formation slots with transparent, editable weights | Know fitness, injuries, training, tactics instructions or dressing-room context |
| Enforce lineup constraints and explain omissions | Prove a lineup or substitution is optimal or caused a result |
| Reconstruct who was on the pitch and the score at any minute | Measure leadership, motivation or "managerial skill" |
| Describe managers' observable timing / rotation / formation patterns | Rank managers from 1-2 matches |
| Fit a leakage-free, gated model of observed selection behaviour | Forecast results or player value |

**Key terms.** *Event data* (time-stamped on-ball actions) vs *aggregated statistics*; *per-90*; *percentile
within role*; *xG* (StatsBomb model probability that a shot is scored — provider output); *game state*; *data leakage*;
*chronological split*; *association vs causation*; *Hungarian assignment*.

Full viva question bank with answer prompts: `reports/VIVA_PREPARATION.md`. Presentation outline and live-demo
sequence: same file. Review evidence templates: `reports/REVIEW_1_DATA_COLLECTION.md`,
`reports/REVIEW_2_ANALYSIS_AND_VISUALIZATION.md`.

In [51]:
facts = {"Competition": f"{T['matches'].competition_name.iloc[0]} {T['matches'].season_name.iloc[0]}",
         "Matches analysed": len(T["matches"]), "Events": len(T["events"]), "Teams": len(teams),
         "Players (squads)": T["appearances"].player_id.nunique(), "Substitutions": len(S),
         "Quality checks passed": f"{int((B.quality.status == 'PASS').sum())}/{len(B.quality)}",
         "Total pipeline time (s)": round(parse_seconds + B.timings['features_s'], 1)}
display(Markdown("**Numbers to quote in the viva (from this run)**"))
display(pd.DataFrame(list(facts.items()), columns=["fact", "value"]))
status_badge()

**Numbers to quote in the viva (from this run)**

,fact,value
0,Competition,1. Bundesliga 2023/2024
1,Matches analysed,34
2,Events,137765
3,Teams,18
4,Players (squads),460
5,Substitutions,301
6,Quality checks passed,16/16
7,Total pipeline time (s),6.6
